# FinQA Evaluation Analysis

Analysis of dev set evaluation run `run_20260213_140849_dev_883` — 883 examples evaluated with gpt-5-nano.

Sections:
1. Setup & Data Loading
2. Overall Accuracy
3. Error Breakdown & Failure Analysis
4. Self-Correction & Retries
5. Per-Node Latency Analysis
6. LLM Token Usage
7. End-to-End Latency Distribution
8. Numerical Reasoning Complexity
9. Accuracy by Question Type
10. Summary

## 1. Setup & Data Loading

In [ ]:
import sys
import json
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Project imports
sys.path.insert(0, str(Path('..').resolve()))
from dotenv import load_dotenv
load_dotenv(Path('..') / '.env', override=True)

from finqa_chatbot.storage import get_mongo_store

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

RUN_ID = 'run_20260213_140849_dev_883'

In [ ]:
# Connect to MongoDB and load all data
store = get_mongo_store()
assert store is not None, 'MongoDB not available — check MONGODB_URI in .env'

# Load run metadata
run_info = store.get_run(RUN_ID)
assert run_info is not None, f'Run {RUN_ID} not found'
print(f"Run: {RUN_ID}")
print(f"Split: {run_info.get('split')}")
print(f"Examples: {run_info.get('num_examples')}")
dur = run_info.get('duration_seconds')
print(f"Duration: {dur:.0f}s" if dur is not None else "Duration: n/a")
cfg = run_info.get('config') or {}
print(f"Config: {json.dumps(cfg, indent=2)}")
print(f"\nOfficial results:")
for k, v in (run_info.get('results_official') or {}).items():
    print(f"  {k}: {v}")

In [ ]:
# Load predictions and traces into DataFrames
preds_raw = store.get_predictions(RUN_ID)
traces_raw = list(store.traces.find({'run_id': RUN_ID}, {'_id': 0}))

preds_df = pd.DataFrame(preds_raw)
traces_df = pd.DataFrame(traces_raw)

print(f"Predictions: {len(preds_df)} rows")
print(f"Trace steps: {len(traces_df)} rows")
print(f"\nPrediction columns: {list(preds_df.columns)}")
print(f"Trace columns: {list(traces_df.columns)}")

## 2. Overall Accuracy

In [ ]:
total = len(preds_df)
exe_correct = preds_df['exe_correct'].sum()
prog_correct = preds_df['prog_correct'].sum()
exe_acc = exe_correct / total
prog_acc = prog_correct / total

# Count categories
has_error = preds_df['error'].notna() & (preds_df['error'] != '')
no_program = preds_df['raw_program'].isna() | (preds_df['raw_program'] == '')

print(f"{'Metric':<30} {'Value':>10} {'Count':>10}")
print('-' * 52)
print(f"{'Total examples':<30} {'':>10} {total:>10}")
print(f"{'Execution accuracy (exe_acc)':<30} {exe_acc:>9.1%} {exe_correct:>10.0f}")
print(f"{'Program accuracy (prog_acc)':<30} {prog_acc:>9.1%} {prog_correct:>10.0f}")
print(f"{'No program produced':<30} {'':>10} {no_program.sum():>10}")
print(f"{'Errors during execution':<30} {'':>10} {has_error.sum():>10}")

# Accuracy bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart: correct vs incorrect
exe_fail = total - exe_correct
axes[0].pie([exe_correct, exe_fail],
            labels=[f'Correct ({exe_correct})', f'Incorrect ({exe_fail:.0f})'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%', startangle=90)
axes[0].set_title('Execution Accuracy')

# Bar chart: exe_acc vs prog_acc
bars = axes[1].bar(['exe_acc', 'prog_acc'], [exe_acc, prog_acc],
                    color=['#3498db', '#9b59b6'], width=0.5)
for bar, val in zip(bars, [exe_acc, prog_acc]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.1%}', ha='center', fontweight='bold')
axes[1].set_ylim(0, 1)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1].set_title('Accuracy Metrics')
axes[1].set_ylabel('Accuracy')

plt.tight_layout()
plt.show()

## 3. Error Breakdown & Failure Analysis

In [ ]:
# Categorize failures
failures = preds_df[~preds_df['exe_correct']].copy()
print(f"Total failures: {len(failures)}")
print()

# Classify failure types
def classify_failure(row):
    prog = row.get('raw_program', '') or ''
    error = row.get('error')
    exe_res = row.get('exe_result')
    gold_ans = row.get('gold_answer')
    
    if not prog:
        return 'no_program'
    if error and ('invalid' in str(error).lower() or 'parse' in str(error).lower()):
        return 'invalid_program'
    if exe_res is None or str(exe_res) == 'invalid_program':
        return 'invalid_program'
    
    # Check if it's a sign/scale error
    try:
        p, g = float(exe_res), float(gold_ans)
        if g != 0:
            ratio = p / g
            if abs(abs(ratio) - 100) < 2 or abs(abs(ratio) - 0.01) < 0.001:
                return 'scale_error'
            if abs(ratio + 1) < 0.05:
                return 'sign_error'
    except (ValueError, TypeError, ZeroDivisionError):
        pass
    
    # Check LLM evaluation if available
    llm_reason = row.get('failure_reason')
    if llm_reason and pd.notna(llm_reason):
        return str(llm_reason)
    
    return 'wrong_answer'

failures['failure_type'] = failures.apply(classify_failure, axis=1)
type_counts = failures['failure_type'].value_counts()

print(f"{'Failure Category':<25} {'Count':>6} {'% of failures':>14} {'% of total':>12}")
print('-' * 60)
for cat, count in type_counts.items():
    print(f"{cat:<25} {count:>6} {count/len(failures):>13.1%} {count/total:>11.1%}")

# Failure breakdown chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.Set2(np.linspace(0, 1, len(type_counts)))
bars = ax.barh(type_counts.index, type_counts.values, color=colors)
for bar, val in zip(bars, type_counts.values):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{val} ({val/total:.1%})', va='center')
ax.set_xlabel('Count')
ax.set_title(f'Failure Categories ({len(failures)} failures out of {total})')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# LLM evaluation results (if available)
llm_evaluated = preds_df[preds_df['failure_reason'].notna() & (preds_df['failure_reason'] != '')]

if len(llm_evaluated) > 0:
    print(f"LLM-evaluated predictions: {len(llm_evaluated)}")
    llm_correct = llm_evaluated[llm_evaluated['llm_correct'] == True]
    print(f"LLM judged correct (false negatives rescued): {len(llm_correct)}")
    print()
    reason_counts = llm_evaluated['failure_reason'].value_counts()
    print(f"{'Failure Reason':<25} {'Count':>6}")
    print('-' * 33)
    for reason, count in reason_counts.items():
        print(f"{reason:<25} {count:>6}")
else:
    print('No LLM evaluation data available. Run: python scripts/query_results.py evaluate', RUN_ID)

## 4. Self-Correction & Retries

In [ ]:
rounds_used = preds_df['rounds_used'].fillna(1).astype(int)
round_counts = rounds_used.value_counts().sort_index()

print(f"Average rounds used: {rounds_used.mean():.2f}")
print(f"Max rounds used: {rounds_used.max()}")
print(f"Examples needing >1 round: {(rounds_used > 1).sum()} ({(rounds_used > 1).mean():.1%})")
print()

print(f"{'Rounds':>6} {'Count':>8} {'% of total':>12} {'exe_acc':>10}")
print('-' * 40)
for r in sorted(round_counts.index):
    mask = rounds_used == r
    count = mask.sum()
    acc = preds_df.loc[mask, 'exe_correct'].mean()
    print(f"{r:>6} {count:>8} {count/total:>11.1%} {acc:>9.1%}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution of rounds used
axes[0].bar(round_counts.index, round_counts.values, color='#3498db', edgecolor='white')
axes[0].set_xlabel('Rounds Used')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Self-Correction Rounds')
for i, (r, c) in enumerate(round_counts.items()):
    axes[0].text(r, c + 5, str(c), ha='center', fontsize=9)

# Accuracy by round count
round_accs = []
round_labels = []
for r in sorted(round_counts.index):
    mask = rounds_used == r
    if mask.sum() >= 5:  # only show if enough samples
        round_accs.append(preds_df.loc[mask, 'exe_correct'].mean())
        round_labels.append(f'Round {r}\n(n={mask.sum()})')

bars = axes[1].bar(round_labels, round_accs, color='#2ecc71', edgecolor='white')
for bar, val in zip(bars, round_accs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.1%}', ha='center', fontsize=9)
axes[1].set_ylim(0, 1)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1].set_title('Execution Accuracy by Rounds Used')
axes[1].set_ylabel('exe_acc')

plt.tight_layout()
plt.show()

In [ ]:
# Verification status breakdown
if 'verification_status' in preds_df.columns:
    vstatus = preds_df['verification_status'].fillna('unknown').value_counts()
    print('Verification status distribution:')
    for status, count in vstatus.items():
        acc = preds_df.loc[preds_df['verification_status'] == status, 'exe_correct'].mean()
        print(f"  {status:<20} {count:>5} ({count/total:.1%})  exe_acc={acc:.1%}")

    # Self-correction effectiveness
    multi_round = preds_df[rounds_used > 1]
    if len(multi_round) > 0:
        print(f"\nSelf-correction analysis:")
        print(f"  Examples that needed retry: {len(multi_round)}")
        print(f"  Of those, exe_correct: {multi_round['exe_correct'].sum()} ({multi_round['exe_correct'].mean():.1%})")
        single_round = preds_df[rounds_used == 1]
        print(f"  Single-round exe_acc: {single_round['exe_correct'].mean():.1%} (n={len(single_round)})")
        print(f"  Multi-round exe_acc:  {multi_round['exe_correct'].mean():.1%} (n={len(multi_round)})")

## 5. Per-Node Latency Analysis

In [ ]:
if len(traces_df) > 0:
    # Per-node latency stats
    node_stats = traces_df.groupby('node_name')['duration_ms'].agg(
        count='count',
        mean='mean',
        median='median',
        p95=lambda x: np.percentile(x.dropna(), 95) if len(x.dropna()) > 0 else 0,
        max='max',
        total='sum',
    ).round(1)
    node_stats = node_stats.sort_values('mean', ascending=False)

    print(f"{'Node':<16} {'Count':>6} {'Mean ms':>9} {'Median ms':>10} {'P95 ms':>9} {'Max ms':>9} {'Total s':>9}")
    print('-' * 72)
    for node, row in node_stats.iterrows():
        print(f"{node:<16} {row['count']:>6.0f} {row['mean']:>9.1f} {row['median']:>10.1f} "
              f"{row['p95']:>9.1f} {row['max']:>9.1f} {row['total']/1000:>9.1f}")

    # Bar chart: mean latency per node
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    nodes_sorted = node_stats.sort_values('mean', ascending=True)
    colors = ['#e74c3c' if n == 'summarizer' else '#3498db' for n in nodes_sorted.index]
    axes[0].barh(nodes_sorted.index, nodes_sorted['mean'], color=colors)
    axes[0].set_xlabel('Mean Latency (ms)')
    axes[0].set_title('Mean Latency per Node')
    for i, (node, row) in enumerate(nodes_sorted.iterrows()):
        axes[0].text(row['mean'] + 20, i, f"{row['mean']:.0f}ms", va='center', fontsize=8)

    # Total time share per node
    total_time = node_stats['total'].sum()
    node_share = (node_stats['total'] / total_time * 100).sort_values(ascending=False)
    colors_pie = plt.cm.Set2(np.linspace(0, 1, len(node_share)))
    wedges, texts, autotexts = axes[1].pie(
        node_share.values, labels=node_share.index,
        autopct=lambda pct: f'{pct:.1f}%' if pct > 3 else '',
        colors=colors_pie, startangle=90
    )
    axes[1].set_title('Share of Total Pipeline Time')

    plt.tight_layout()
    plt.show()
else:
    print('No trace data available.')

In [ ]:
if len(traces_df) > 0:
    # Latency distribution for the top 3 slowest nodes
    top_nodes = node_stats.head(3).index.tolist()
    fig, axes = plt.subplots(1, len(top_nodes), figsize=(5 * len(top_nodes), 4))
    if len(top_nodes) == 1:
        axes = [axes]

    for ax, node in zip(axes, top_nodes):
        data = traces_df.loc[traces_df['node_name'] == node, 'duration_ms'].dropna()
        ax.hist(data.to_numpy(), bins=40, color='#3498db', alpha=0.7, edgecolor='white')
        ax.axvline(data.mean(), color='red', linestyle='--', label=f'Mean: {data.mean():.0f}ms')
        ax.axvline(data.median(), color='orange', linestyle='--', label=f'Median: {data.median():.0f}ms')
        ax.set_xlabel('Duration (ms)')
        ax.set_ylabel('Count')
        ax.set_title(f'{node} latency (n={len(data)})')
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

## 6. LLM Token Usage

In [ ]:
if len(traces_df) > 0 and 'llm_usage' in traces_df.columns:
    # Extract LLM usage from traces
    llm_traces = traces_df[traces_df['llm_usage'].notna()].copy()
    
    if len(llm_traces) > 0:
        llm_traces['prompt_tokens'] = llm_traces['llm_usage'].apply(
            lambda x: x.get('total_prompt_tokens', 0) if isinstance(x, dict) else 0)
        llm_traces['completion_tokens'] = llm_traces['llm_usage'].apply(
            lambda x: x.get('total_completion_tokens', 0) if isinstance(x, dict) else 0)
        llm_traces['total_tokens'] = llm_traces['llm_usage'].apply(
            lambda x: x.get('total_tokens', 0) if isinstance(x, dict) else 0)
        llm_traces['num_calls'] = llm_traces['llm_usage'].apply(
            lambda x: x.get('num_calls', 0) if isinstance(x, dict) else 0)

        # Per-node LLM usage
        llm_by_node = llm_traces.groupby('node_name').agg(
            total_calls=('num_calls', 'sum'),
            total_prompt=('prompt_tokens', 'sum'),
            total_completion=('completion_tokens', 'sum'),
            total_tokens=('total_tokens', 'sum'),
            avg_prompt=('prompt_tokens', 'mean'),
            avg_completion=('completion_tokens', 'mean'),
        ).round(0).astype(int)
        llm_by_node = llm_by_node.sort_values('total_tokens', ascending=False)

        print(f"{'Node':<16} {'Calls':>7} {'Tot Prompt':>11} {'Tot Compl':>11} {'Tot Tokens':>11} {'Avg Prompt':>11} {'Avg Compl':>11}")
        print('-' * 82)
        for node, row in llm_by_node.iterrows():
            print(f"{node:<16} {row['total_calls']:>7} {row['total_prompt']:>11,} {row['total_completion']:>11,} "
                  f"{row['total_tokens']:>11,} {row['avg_prompt']:>11,} {row['avg_completion']:>11,}")

        grand_total = llm_by_node['total_tokens'].sum()
        grand_prompt = llm_by_node['total_prompt'].sum()
        grand_completion = llm_by_node['total_completion'].sum()
        print(f"\nGrand total: {grand_total:,} tokens ({grand_prompt:,} prompt + {grand_completion:,} completion)")
        print(f"Average per example: {grand_total / total:,.0f} tokens ({grand_prompt / total:,.0f}p + {grand_completion / total:,.0f}c)")

        # Cost estimate (gpt-5-nano pricing estimate)
        # Approximate: $0.10/1M input, $0.40/1M output
        cost_input = grand_prompt / 1_000_000 * 0.10
        cost_output = grand_completion / 1_000_000 * 0.40
        print(f"\nEstimated cost: ${cost_input + cost_output:.3f} total (${cost_input:.3f} input + ${cost_output:.3f} output)")
        print(f"Cost per example: ${(cost_input + cost_output) / total:.5f}")

        # Token distribution visualization
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # Stacked bar: prompt vs completion per node
        nodes = llm_by_node.index.tolist()
        x = np.arange(len(nodes))
        axes[0].bar(x, llm_by_node['total_prompt'], label='Prompt', color='#3498db')
        axes[0].bar(x, llm_by_node['total_completion'], bottom=llm_by_node['total_prompt'],
                    label='Completion', color='#e74c3c')
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(nodes, rotation=45, ha='right')
        axes[0].set_ylabel('Tokens')
        axes[0].set_title('Token Usage by Node')
        axes[0].legend()

        # Per-example total token distribution
        per_entry_tokens = llm_traces.groupby('entry_id')['total_tokens'].sum()
        axes[1].hist(per_entry_tokens.to_numpy(), bins=40, color='#9b59b6', alpha=0.7, edgecolor='white')
        axes[1].axvline(per_entry_tokens.mean(), color='red', linestyle='--',
                        label=f'Mean: {per_entry_tokens.mean():,.0f}')
        axes[1].set_xlabel('Total Tokens per Example')
        axes[1].set_ylabel('Count')
        axes[1].set_title('Token Usage Distribution per Example')
        axes[1].legend()

        plt.tight_layout()
        plt.show()
    else:
        print('No LLM usage data in traces.')
else:
    print('No trace data or llm_usage column available.')

## 7. End-to-End Latency Distribution

In [ ]:
if len(traces_df) > 0:
    # Compute end-to-end latency per entry
    e2e = traces_df.groupby('entry_id')['duration_ms'].sum().rename('total_ms')
    e2e_seconds = e2e / 1000

    print(f"End-to-end latency per example:")
    print(f"  Mean:   {e2e_seconds.mean():.2f}s")
    print(f"  Median: {e2e_seconds.median():.2f}s")
    print(f"  P95:    {np.percentile(e2e_seconds.dropna(), 95):.2f}s")
    print(f"  P99:    {np.percentile(e2e_seconds.dropna(), 99):.2f}s")
    print(f"  Max:    {e2e_seconds.max():.2f}s")
    print(f"  Min:    {e2e_seconds.min():.2f}s")

    # Merge with correctness
    e2e_df = e2e_seconds.reset_index()
    e2e_df.columns = ['entry_id', 'latency_s']
    e2e_df = e2e_df.merge(preds_df[['entry_id', 'exe_correct', 'rounds_used']], on='entry_id', how='left')

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Overall latency histogram
    axes[0].hist(e2e_df['latency_s'].to_numpy(), bins=50, color='#3498db', alpha=0.7, edgecolor='white')
    axes[0].axvline(e2e_df['latency_s'].mean(), color='red', linestyle='--',
                    label=f'Mean: {e2e_df["latency_s"].mean():.1f}s')
    axes[0].axvline(e2e_df['latency_s'].median(), color='orange', linestyle='--',
                    label=f'Median: {e2e_df["latency_s"].median():.1f}s')
    axes[0].set_xlabel('Latency (seconds)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('End-to-End Latency Distribution')
    axes[0].legend(fontsize=8)

    # Latency by correctness
    correct_latency = e2e_df.loc[e2e_df['exe_correct'] == True, 'latency_s']
    incorrect_latency = e2e_df.loc[e2e_df['exe_correct'] == False, 'latency_s']
    axes[1].boxplot([correct_latency.dropna().to_numpy(), incorrect_latency.dropna().to_numpy()],
                    labels=['Correct', 'Incorrect'], vert=True)
    axes[1].set_ylabel('Latency (seconds)')
    axes[1].set_title('Latency: Correct vs Incorrect')

    # Latency by rounds used
    round_groups = []
    round_labels = []
    for r in sorted(e2e_df['rounds_used'].dropna().unique()):
        r = int(r)
        data = e2e_df.loc[e2e_df['rounds_used'] == r, 'latency_s'].dropna()
        if len(data) >= 3:
            round_groups.append(data.to_numpy())
            round_labels.append(f'R{r} (n={len(data)})')
    if round_groups:
        axes[2].boxplot(round_groups, labels=round_labels, vert=True)
        axes[2].set_ylabel('Latency (seconds)')
        axes[2].set_title('Latency by Rounds Used')

    plt.tight_layout()
    plt.show()
else:
    print('No trace data available.')

## 8. Numerical Reasoning Complexity

In [ ]:
# Analyze predicted vs gold programs
OP_RE = re.compile(r'([a-z_]+)\s*\(')

def count_steps(prog):
    if not prog or pd.isna(prog):
        return 0
    return len(OP_RE.findall(str(prog)))

def extract_ops(prog):
    if not prog or pd.isna(prog):
        return []
    return OP_RE.findall(str(prog))

preds_df['gold_steps'] = preds_df['gold_program'].apply(count_steps)
preds_df['pred_steps'] = preds_df['raw_program'].apply(count_steps)
preds_df['gold_ops'] = preds_df['gold_program'].apply(extract_ops)

# Accuracy by gold program complexity
print(f"{'Gold Steps':>10} {'Count':>7} {'exe_acc':>9} {'prog_acc':>10}")
print('-' * 40)
for steps in sorted(preds_df['gold_steps'].unique()):
    if steps == 0:
        continue
    mask = preds_df['gold_steps'] == steps
    count = mask.sum()
    ea = preds_df.loc[mask, 'exe_correct'].mean()
    pa = preds_df.loc[mask, 'prog_correct'].mean()
    print(f"{steps:>10} {count:>7} {ea:>8.1%} {pa:>9.1%}")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Gold program step distribution
step_dist = preds_df['gold_steps'].value_counts().sort_index()
step_dist = step_dist[step_dist.index > 0]
axes[0].bar(step_dist.index, step_dist.values, color='#3498db', edgecolor='white')
axes[0].set_xlabel('Number of Steps')
axes[0].set_ylabel('Count')
axes[0].set_title('Gold Program Complexity')
for x, y in zip(step_dist.index, step_dist.values):
    axes[0].text(x, y + 3, str(y), ha='center', fontsize=9)

# Accuracy by complexity
steps_range = sorted([s for s in preds_df['gold_steps'].unique() if s > 0])
exe_by_step = [preds_df.loc[preds_df['gold_steps'] == s, 'exe_correct'].mean() for s in steps_range]
prog_by_step = [preds_df.loc[preds_df['gold_steps'] == s, 'prog_correct'].mean() for s in steps_range]
x = np.arange(len(steps_range))
w = 0.35
axes[1].bar(x - w/2, exe_by_step, w, label='exe_acc', color='#2ecc71')
axes[1].bar(x + w/2, prog_by_step, w, label='prog_acc', color='#9b59b6')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'{s} step' for s in steps_range])
axes[1].set_ylim(0, 1)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1].set_title('Accuracy by Program Complexity')
axes[1].legend()

# Predicted vs gold step count
valid_mask = (preds_df['gold_steps'] > 0) & (preds_df['pred_steps'] > 0)
axes[2].scatter(preds_df.loc[valid_mask, 'gold_steps'],
                preds_df.loc[valid_mask, 'pred_steps'],
                alpha=0.3, s=20, c=preds_df.loc[valid_mask, 'exe_correct'].map({True: '#2ecc71', False: '#e74c3c'}))
max_step = max(preds_df['gold_steps'].max(), preds_df['pred_steps'].max()) + 1
axes[2].plot([0, max_step], [0, max_step], 'k--', alpha=0.3, label='y=x')
axes[2].set_xlabel('Gold Steps')
axes[2].set_ylabel('Predicted Steps')
axes[2].set_title('Predicted vs Gold Complexity\n(green=correct, red=incorrect)')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Accuracy by operation type
all_ops = set()
for ops in preds_df['gold_ops']:
    all_ops.update(ops)

print(f"{'Operation':<20} {'Entries':>8} {'exe_acc':>9} {'prog_acc':>10}")
print('-' * 50)
op_data = []
for op in sorted(all_ops):
    mask = preds_df['gold_ops'].apply(lambda ops: op in ops)
    count = mask.sum()
    if count < 3:
        continue
    ea = preds_df.loc[mask, 'exe_correct'].mean()
    pa = preds_df.loc[mask, 'prog_correct'].mean()
    op_data.append({'op': op, 'count': count, 'exe_acc': ea, 'prog_acc': pa})
    print(f"{op:<20} {count:>8} {ea:>8.1%} {pa:>9.1%}")

# Bar chart
if op_data:
    op_df = pd.DataFrame(op_data).sort_values('count', ascending=False)
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(op_df))
    w = 0.35
    ax.bar(x - w/2, op_df['exe_acc'], w, label='exe_acc', color='#2ecc71')
    ax.bar(x + w/2, op_df['prog_acc'], w, label='prog_acc', color='#9b59b6')
    ax.set_xticks(x)
    ax.set_xticklabels([f"{r['op']}\n(n={r['count']})" for _, r in op_df.iterrows()],
                       rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.set_title('Accuracy by Gold Operation Type')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 9. Accuracy by Question Type

In [ ]:
# Load gold data to get questions
import json
dev_path = Path('../data/dev.json')
with open(dev_path) as f:
    dev_data = json.load(f)
gold_dict = {e['id']: e for e in dev_data}

# Classify questions
def classify_question(entry_id):
    entry = gold_dict.get(entry_id)
    if not entry:
        return 'unknown'
    q = entry['qa']['question'].lower()
    if 'percent change' in q or 'percentage change' in q:
        return 'percent_change'
    if 'what percentage' in q or 'what portion' in q or 'what percent' in q:
        return 'what_percentage'
    if 'change' in q or 'difference' in q or 'net change' in q:
        return 'change_difference'
    if 'ratio' in q:
        return 'ratio'
    if 'total' in q or 'sum' in q or 'combined' in q:
        return 'total_sum'
    if 'average' in q or 'mean' in q:
        return 'average'
    if 'by how much' in q:
        return 'by_how_much'
    if 'roi' in q or 'return on investment' in q or 'cumulative' in q:
        return 'roi_cumulative'
    return 'other'

preds_df['question_type'] = preds_df['entry_id'].apply(classify_question)

qtype_stats = preds_df.groupby('question_type').agg(
    count=('exe_correct', 'count'),
    exe_acc=('exe_correct', 'mean'),
    prog_acc=('prog_correct', 'mean'),
).sort_values('count', ascending=False)

print(f"{'Question Type':<20} {'Count':>7} {'exe_acc':>9} {'prog_acc':>10}")
print('-' * 50)
for qtype, row in qtype_stats.iterrows():
    print(f"{qtype:<20} {row['count']:>7.0f} {row['exe_acc']:>8.1%} {row['prog_acc']:>9.1%}")

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(qtype_stats))
w = 0.35
ax.bar(x - w/2, qtype_stats['exe_acc'], w, label='exe_acc', color='#2ecc71')
ax.bar(x + w/2, qtype_stats['prog_acc'], w, label='prog_acc', color='#9b59b6')
ax.set_xticks(x)
ax.set_xticklabels([f"{qt}\n(n={int(qtype_stats.loc[qt, 'count'])})" for qt in qtype_stats.index],
                   rotation=45, ha='right', fontsize=8)
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_title('Accuracy by Question Type')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Summary

In [ ]:
print('=' * 60)
print('EVALUATION SUMMARY')
print('=' * 60)
print(f"Run:     {RUN_ID}")
print(f"Model:   {(run_info.get('config') or {}).get('model_name', 'unknown')}")
print(f"Split:   {run_info.get('split')} ({total} examples)")
dur = run_info.get('duration_seconds')
print(f"Duration: {dur:.0f}s" if dur is not None else "Duration: n/a")
print()
print('--- Accuracy ---')
print(f"  exe_acc:  {exe_acc:.1%} ({exe_correct:.0f}/{total})")
print(f"  prog_acc: {prog_acc:.1%} ({prog_correct:.0f}/{total})")
print()
print('--- Retries ---')
print(f"  Avg rounds: {rounds_used.mean():.2f}")
print(f"  Multi-round: {(rounds_used > 1).sum()} ({(rounds_used > 1).mean():.1%})")
print()

if len(traces_df) > 0:
    e2e = traces_df.groupby('entry_id')['duration_ms'].sum() / 1000
    print('--- Latency ---')
    print(f"  Mean:   {e2e.mean():.2f}s")
    print(f"  Median: {e2e.median():.2f}s")
    print(f"  P95:    {np.percentile(e2e.dropna(), 95):.2f}s")
    print()

    if 'llm_usage' in traces_df.columns:
        llm_t = traces_df[traces_df['llm_usage'].notna()]
        if len(llm_t) > 0:
            total_tok = llm_t['llm_usage'].apply(
                lambda x: x.get('total_tokens', 0) if isinstance(x, dict) else 0).sum()
            print('--- LLM Usage ---')
            print(f"  Total tokens: {total_tok:,}")
            print(f"  Avg per example: {total_tok / total:,.0f}")
            print()

print('--- Failure Breakdown ---')
for cat, count in type_counts.items():
    print(f"  {cat}: {count}")

print()
print('--- Hardest Question Types ---')
worst = qtype_stats[qtype_stats['count'] >= 10].sort_values('exe_acc').head(3)
for qt, row in worst.iterrows():
    ea = row['exe_acc']
    cnt = row['count']
    print(f"  {qt}: {ea:.1%} exe_acc (n={cnt:.0f})")

print()
print('--- Key Findings ---')
if len(traces_df) > 0:
    slowest = node_stats.index[0]
    slowest_ms = node_stats.loc[slowest, 'mean']
    print(f"  Slowest node: {slowest} ({slowest_ms:.0f}ms avg)")
single_acc = preds_df.loc[preds_df['gold_steps'] == 1, 'exe_correct'].mean()
print(f"  Single-step accuracy: {single_acc:.1%}")
multi = preds_df.loc[preds_df['gold_steps'] >= 3, 'exe_correct'].mean()
print(f"  Multi-step (>=3) accuracy: {multi:.1%}")